In [1]:
#| echo: false
#| warning: false
#| output: asis

from IPython.display import Markdown, display

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from statsmodels.formula.api import ols
from statsmodels.iolib.summary2 import summary_col

# Не показывать Warning
import warnings
warnings.simplefilter(action='ignore', category=Warning)

# Настройки оформления графиков
plt.rcParams['figure.figsize'] = (8.5, 6)
plt.rcParams['font.size'] = 10
plt.rcParams['axes.labelsize'] = 12
plt.rcParams['axes.titlesize'] = 14

CONST_NAMES = ('const', 'Intercept')


def fmt(x, decimals=3):
    """Число с заданным числом знаков; оценка, которая округляется до нуля, — с тремя значащими цифрами."""
    if pd.isna(x):
        return ''
    text = f'{x:.{decimals}f}'
    if float(text) == 0:
        return f'{x:.3g}' if x != 0 else f'{0:.{decimals}f}'
    return text


def fmt_p(p, decimals=3):
    """P-значение: меньше 0.001 печатается как «<0.001»."""
    return '<0.001' if p < 0.001 else f'{p:.{decimals}f}'


def errors_text(res, case='nom'):
    """Название типа стандартных ошибок подгонки: 'nom' — «неробастные», 'dat' — «по неробастным …»."""
    cov_type = getattr(res, 'cov_type', 'nonrobust')
    if cov_type == 'nonrobust':
        return {'nom': 'неробастные стандартные ошибки',
                'dat': 'по неробастным стандартным ошибкам'}[case]
    return {'nom': f'робастные стандартные ошибки, {cov_type}',
            'dat': f'по робастным стандартным ошибкам {cov_type}'}[case]


def fit_ols(formula, data, robust=False, **kwargs):
    """МНК-подгонка; robust=True — робастные ошибки HC3. Всегда t- и F-распределения (use_t=True)."""
    cov_type = 'HC3' if robust else 'nonrobust'
    return ols(formula, data=data).fit(cov_type=cov_type, use_t=True, **kwargs)


def show_table(df, index=True, align=None):
    """Таблица pandas в Markdown; числа должны быть заранее превращены в строки через fmt."""
    display(Markdown('\n\n' + df.to_markdown(index=index, colalign=align, disable_numparse=True) + '\n\n'))


def nobs_text(res):
    """Строка «Модель была подогнана по N наблюдениям»."""
    n = int(res.nobs)
    word = 'наблюдению' if n % 10 == 1 and n % 100 != 11 else 'наблюдениям'
    return f'Модель была подогнана по {n} {word}.'


def show_coefs(res, decimals=3, show_nobs=True):
    """Оценки коэффициентов: вертикальная таблица «Переменная | Оценка» и число наблюдений."""
    cov_type = getattr(res, 'cov_type', 'nonrobust')
    column = 'Оценка' if cov_type == 'nonrobust' else f'Оценка ({cov_type})'
    table = pd.DataFrame({'Переменная': res.params.index,
                          column: [fmt(b, decimals) for b in res.params]})
    text = table.to_markdown(index=False, colalign=('left', 'right'), disable_numparse=True)
    if show_nobs:
        text += '\n\n' + nobs_text(res)
    display(Markdown('\n\n' + text + '\n\n'))


def display_answer(text, use_callout=True, theme='note', title='Критическое значение'):
    """Ответ в рамке (callout): theme='note' — число, 'tip' — вывод, 'important' — условие."""
    if use_callout:
        head = f'title="{title}" ' if title else ''
        output_md = f'\n\n::: {{.callout-{theme} {head}icon="true"}}\n{text}\n:::\n\n'
    elif title:
        output_md = f'\n\n**{title}:** {text}\n\n'
    else:
        output_md = f'\n\n{text}\n\n'
    display(Markdown(output_md))


def show_conditions(level, errors=None, hypothesis=None, extra=()):
    """Блок «Условия теста»: уровень значимости (доля или список долей), тип ошибок, гипотеза формулой."""
    levels = level if isinstance(level, (list, tuple)) else [level]
    items = ['уровень значимости: **' + ' и '.join(f'{a * 100:g}%' for a in levels) + '**']
    if errors:
        items.append(f'стандартные ошибки: **{errors}**')
    hypotheses = [hypothesis] if isinstance(hypothesis, str) else list(hypothesis or [])
    if len(hypotheses) == 1:
        items.append(f'гипотеза: {hypotheses[0]}')
    else:
        items += [f'гипотеза {i}: {h}' for i, h in enumerate(hypotheses, start=1)]
    items += list(extra)
    display_answer('\n'.join(f'- {item}' for item in items), theme='important', title='Условия теста')


def signif_stars(p):
    """Коды значимости: *** p<0.01, ** p<0.05, * p<0.1."""
    return '***' if p < 0.01 else '**' if p < 0.05 else '*' if p < 0.1 else ''


SIGNIF_LEGEND = '*Signif. codes:* `***` $p<0.01$, `**` $p<0.05$, `*` $p<0.1$'


def print_regression_summary(res, decimals=3, show_signif=True):
    """Таблица t-теста: Coef., Std.Err., t-value, p-value, Signif. Заголовок называет тип ошибок."""
    title = f'**Результаты $t$-теста для коэффициентов ({errors_text(res)}):**'
    table = pd.DataFrame({
        'Coef.': [fmt(b, decimals) for b in res.params],
        'Std.Err.': [fmt(s, decimals) for s in res.bse],
        't-value': [fmt(t, decimals) for t in res.tvalues],
        'p-value': [fmt_p(p, decimals) for p in res.pvalues],
    }, index=res.params.index)
    legend = ''
    if show_signif:
        table['Signif.'] = ['' if name in CONST_NAMES else signif_stars(p)
                            for name, p in res.pvalues.items()]
        legend = '\n\n' + SIGNIF_LEGEND
    align = ('left',) + ('right',) * 4 + (('left',) if show_signif else ())
    display(Markdown(f'\n\n{title}\n\n' + table.to_markdown(colalign=align, disable_numparse=True) + legend + '\n\n'))


def print_significant_coeffs(res, sign_level=0.05, theme='note', title=None):
    """Рамка со списком коэффициентов, значимых на уровне sign_level (без константы)."""
    names = [name for name, p in res.pvalues.items() if p < sign_level and name not in CONST_NAMES]
    if title is None:
        title = f'Коэффициенты, значимые на уровне {sign_level * 100:g}% ({errors_text(res, "dat")})'
    text = ', '.join(f'`{name}`' for name in names) if names else 'Значимых коэффициентов не выявлено.'
    display_answer(text, theme=theme, title=title)


def show_models(models, names=None, decimals=3, regressor_order=(), errors_note=None, info=None):
    """Сравнение регрессий (summary_col): строка Dep. Variable, коды значимости, пояснение про скобки.

    info — словарь {название строки: функция от подгонки} для дополнительных строк (N, F и т. п.).
    """
    table = summary_col(models, model_names=names, stars=True, float_format=f'%0.{decimals}f',
                        regressor_order=list(regressor_order), info_dict=info or {}).tables[0]
    top = pd.DataFrame({col: [m.model.endog_names] for col, m in zip(table.columns, models)},
                       index=['Dep. Variable'])
    table = pd.concat([top, table])
    if errors_note is None:
        errors_note = errors_text(models[0])
    note = f'{SIGNIF_LEGEND}. В скобках — {errors_note}.'
    display(Markdown('\n\n' + table.to_markdown(disable_numparse=True) + '\n\n' + note + '\n\n'))


def show_ci(res, level=0.95, decimals=3):
    """Доверительные интервалы: «Переменная | Нижняя граница | Верхняя граница», заголовок над таблицей."""
    ci = res.conf_int(alpha=1 - level)
    table = pd.DataFrame({'Переменная': ci.index,
                          'Нижняя граница': [fmt(x, decimals) for x in ci[0]],
                          'Верхняя граница': [fmt(x, decimals) for x in ci[1]]})
    title = f'**{level * 100:g}%-е доверительные интервалы ({errors_text(res)}):**'
    display(Markdown(f'\n\n{title}\n\n'
                     + table.to_markdown(index=False, colalign=('left', 'right', 'right'), disable_numparse=True) + '\n\n'))

# Загрузка данных
sleep75 = pd.read_csv('../datasets/sleep75.csv')
wage1 = pd.read_csv('../datasets/wage1.csv')
wage2 = pd.read_csv('../datasets/wage2.csv')
Labour = pd.read_csv('../datasets/Labour.csv')
Electricity = pd.read_csv('../datasets/Electricity.csv')

# Помощники листка
TESTS = ('Неробастный F-тест', 'Робастный F-тест (HC3)')
ERRORS = 'неробастные и робастные (HC3) — два варианта F-теста'
ERRORS_T = ERRORS + '; t-тест — неробастные'

# Дополнительные строки сравнительных таблиц
N_OBS = {'Observations': lambda m: f'{int(m.nobs)}'}
RSE = {**N_OBS, 'Residual Std. Error': lambda m: f'{np.sqrt(m.scale):.3f}'}
FULL_STATS = {**RSE, 'F Statistic': lambda m: f'{m.fvalue:.3f}' + signif_stars(m.f_pvalue)}


def fit_pair(formula, data):
    """Две подгонки одной модели: с неробастными и с робастными (HC3) ошибками."""
    return fit_ols(formula, data), fit_ols(formula, data, robust=True)


def const_last(models):
    """Порядок строк сравнительной таблицы: регрессоры, затем константа."""
    names = dict.fromkeys(n for m in models for n in m.params.index if n not in CONST_NAMES)
    return list(names) + ['Intercept']


def show_models_exact(models, names, decimals=3, regressor_order=(), info=None):
    """Как общий show_models, но оценка или ошибка, округляющаяся до нуля, печатается через fmt."""
    table = summary_col(models, model_names=names, stars=True, float_format=f'%0.{decimals}f',
                        regressor_order=list(regressor_order), info_dict=info or {}).tables[0]
    for j, m in enumerate(models):
        for i, name in enumerate(table.index):
            if name in m.params.index and table.iloc[i, j]:
                if float(table.iloc[i, j].rstrip('*')) == 0:
                    table.iloc[i, j] = fmt(m.params[name], decimals) + signif_stars(m.pvalues[name])
                if float(table.iloc[i + 1, j].strip('()')) == 0:
                    table.iloc[i + 1, j] = f'({fmt(m.bse[name], decimals)})'
    top = pd.DataFrame({col: [m.model.endog_names] for col, m in zip(table.columns, models)},
                       index=['Dep. Variable'])
    note = f'{SIGNIF_LEGEND}. В скобках — {errors_text(models[0])}.'
    display(Markdown('\n\n' + pd.concat([top, table]).to_markdown() + '\n\n' + note + '\n\n'))


def f_test(res, hyp=None):
    """F-тест линейных ограничений на подгонке res: (F, df1, df2, P-значение).

    hyp=None — значимость регрессии: все коэффициенты, кроме константы, равны нулю.
    Тип ошибок — тот, с которым сделана подгонка (неробастные или HC3).
    """
    if hyp is None:
        hyp = np.eye(len(res.params))[1:]
    test = res.f_test(hyp)
    return float(np.squeeze(test.fvalue)), int(test.df_num), int(test.df_denom), float(test.pvalue)


def shift_hyp(res, dummy):
    """Матрица ограничений: все коэффициенты при слагаемых с переменной dummy равны нулю."""
    return np.eye(len(res.params))[[dummy in n.split(':') for n in res.params.index]]


def show_f_test(res, hyp=None, decimals=3):
    """Результат одного F-теста: таблица из одной строки. Возвращает F-статистику."""
    F, df1, df2, p = f_test(res, hyp)
    table = pd.DataFrame({'F': [fmt(F, decimals)], 'df1': [df1], 'df2': [df2],
                          'p-value': [fmt_p(p, decimals)]})
    show_table(table, index=False, align=('right',) * 4)
    return F


def f_crit(res, hyp, alpha):
    """Критическое значение F-теста; степени свободы — из самого теста."""
    _, df1, df2, _ = f_test(res, hyp)
    return stats.f.ppf(1 - alpha, df1, df2)


def show_f_crit(res, hyp, alpha, decimals=3):
    """Рамка с критическим значением (оно общее для неробастного и робастного теста)."""
    F_cr = f_crit(res, hyp, alpha)
    display_answer(f'$F_{{crit}} = {fmt(F_cr, decimals)}$', theme='note', title='Критическое значение')
    return F_cr


def verdict(F, F_cr, yes='', no='', decimals=3):
    """Сравнение статистики с критическим значением и решение о гипотезе."""
    sign, text, meaning = ('>', 'отвергается', yes) if F > F_cr else ('<', 'не отвергается', no)
    tail = f': {meaning}' if meaning else ''
    return f'$F = {fmt(F, decimals)} {sign} {fmt(F_cr, decimals)}$, гипотеза $H_0$ {text}{tail}.'


def show_conclusion(F_pair, F_cr, yes='', no='', extra=()):
    """Рамка «Вывод» для пары тестов (неробастный, робастный)."""
    lines = [f'- {name}: {verdict(F, F_cr, yes, no)}' for name, F in zip(TESTS, F_pair)]
    lines += [f'- {item}' for item in extra]
    display_answer('\n'.join(lines), theme='tip', title='Вывод')


def show_conclusions(res, tests, alpha):
    """Рамка для нескольких гипотез: критическое значение и вывод по каждой. tests: (hyp, F, F HC3)."""
    lines = []
    for i, (hyp, *F_pair) in enumerate(tests, start=1):
        F_cr = f_crit(res, hyp, alpha)
        lines.append(f'- Гипотеза {i}: $F_{{crit}} = {fmt(F_cr)}$.')
        lines += [f'    - {name}: {verdict(F, F_cr)}' for name, F in zip(TESTS, F_pair)]
    display_answer('\n'.join(lines), theme='tip', title='Критические значения и выводы')


def show_regression_tests(formulas, data, alpha, decimals=2):
    """Значимость серии регрессий: строка на регрессию, оба варианта F-теста."""
    rows = []
    for i, formula in enumerate(formulas, start=1):
        res, res_hc = fit_pair(formula, data)
        F, F_hc, F_cr = f_test(res)[0], f_test(res_hc)[0], f_crit(res, None, alpha)
        rows.append({'Регрессия': f'({i})', 'F': fmt(F, decimals), 'F (HC3)': fmt(F_hc, decimals),
                     'F крит.': fmt(F_cr, decimals),
                     'Вывод': 'значима' if F > F_cr else 'незначима',
                     'Вывод (HC3)': 'значима' if F_hc > F_cr else 'незначима'})
    show_table(pd.DataFrame(rows), index=False,
               align=('left', 'right', 'right', 'right', 'left', 'left'))

# F-тест: значимость регрессии

## sleep equation #1

Для набора данных `sleep75` рассмотрим несколько регрессий. Переменные: `sleep` — ночной сон, минут в неделю; `totwrk` — рабочее время, минут в неделю; `age` — возраст, лет; остальные регрессоры — бинарные переменные.

In [2]:
#| output: asis
sign_level = 0.05
formulas = [
    'sleep ~ totwrk + age + I(age**2) + south + male + smsa + yngkid + marr + union',
    'sleep ~ totwrk + age + I(age**2) + south + male + smsa + yngkid',
    'sleep ~ totwrk + age + I(age**2) + south + male',
    'sleep ~ smsa + yngkid + marr + union']
mod = [fit_ols(f, sleep75) for f in formulas]
names = [f'({i})' for i in range(1, len(mod) + 1)]
show_models(mod, names, regressor_order=const_last(mod), info=N_OBS)



|                | (1)         | (2)         | (3)         | (4)         |
|:---------------|:------------|:------------|:------------|:------------|
| Dep. Variable  | sleep       | sleep       | sleep       | sleep       |
| totwrk         | -0.167***   | -0.168***   | -0.167***   |             |
|                | (0.018)     | (0.018)     | (0.018)     |             |
| age            | -7.631      | -6.581      | -6.255      |             |
|                | (11.363)    | (11.292)    | (11.191)    |             |
| I(age ** 2)    | 0.124       | 0.112       | 0.109       |             |
|                | (0.135)     | (0.134)     | (0.134)     |             |
| south          | 100.608**   | 98.163**    | 114.547***  |             |
|                | (41.969)    | (41.825)    | (40.637)    |             |
| male           | 86.626**    | 92.256***   | 90.457***   |             |
|                | (35.181)    | (34.571)    | (34.257)    |             |
| smsa           | -53.905     | -57.046*    |             | -57.890*    |
|                | (33.199)    | (32.979)    |             | (34.293)    |
| yngkid         | -17.960     | -13.366     |             | -31.138     |
|                | (50.747)    | (50.178)    |             | (50.424)    |
| marr           | 34.984      |             |             | 60.098      |
|                | (42.452)    |             |             | (44.167)    |
| union          | 13.369      |             |             | 7.327       |
|                | (38.225)    |             |             | (40.530)    |
| Intercept      | 3638.289*** | 3650.673*** | 3613.995*** | 3242.521*** |
|                | (223.997)   | (223.367)   | (218.433)   | (44.044)    |
| R-squared      | 0.132       | 0.131       | 0.128       | 0.007       |
| R-squared Adj. | 0.121       | 0.123       | 0.121       | 0.002       |
| Observations   | 706         | 706         | 706         | 706         |

*Signif. codes:* `***` $p<0.01$, `**` $p<0.05$, `*` $p<0.1$. В скобках — неробастные стандартные ошибки.



In [3]:
#| output: asis
show_conditions(sign_level, ERRORS,
                'каждая регрессия незначима, т.е. все её коэффициенты, кроме константы, равны нулю')



::: {.callout-important title="Условия теста" icon="true"}
- уровень значимости: **5%**
- стандартные ошибки: **неробастные и робастные (HC3) — два варианта F-теста**
- гипотеза: каждая регрессия незначима, т.е. все её коэффициенты, кроме константы, равны нулю
:::



Вычислите тестовые статистики неробастного и робастного F-теста и критические значения. Какие из регрессий значимы? **Ответ округлите до 2-х десятичных знаков.**

In [4]:
#| output: asis
show_regression_tests(formulas, sleep75, sign_level)



| Регрессия   |     F |   F (HC3) |   F крит. | Вывод     | Вывод (HC3)   |
|:------------|------:|----------:|----------:|:----------|:--------------|
| (1)         | 11.80 |      9.87 |      1.89 | значима   | значима       |
| (2)         | 15.09 |     12.60 |      2.02 | значима   | значима       |
| (3)         | 20.49 |     16.92 |      2.23 | значима   | значима       |
| (4)         |  1.31 |      1.40 |      2.38 | незначима | незначима     |



## cost equation #1

Для набора данных `Electricity` рассмотрим несколько регрессий. Переменные: `cost` — общие издержки; `q` — выпуск; `pl` — ставка заработной платы; `pk` — индекс цены капитала; `pf` — цена топлива.

In [5]:
#| output: asis
sign_level = 0.01
formulas = [
    'np.log(cost) ~ np.log(q) + np.log(pl) + np.log(pk) + np.log(pf)',
    'np.log(cost) ~ np.log(q)',
    'np.log(cost) ~ np.log(q) + np.log(pk) + np.log(pf)',
    'np.log(cost) ~ np.log(pl) + np.log(pk) + np.log(pf)',
    'np.log(cost) ~ np.log(pl)']
mod = [fit_ols(f, Electricity) for f in formulas]
names = [f'({i})' for i in range(1, len(mod) + 1)]
show_models(mod, names, regressor_order=const_last(mod), info=N_OBS)



|                | (1)          | (2)          | (3)          | (4)          | (5)          |
|:---------------|:-------------|:-------------|:-------------|:-------------|:-------------|
| Dep. Variable  | np.log(cost) | np.log(cost) | np.log(cost) | np.log(cost) | np.log(cost) |
| np.log(q)      | 0.838***     | 0.836***     | 0.839***     |              |              |
|                | (0.009)      | (0.013)      | (0.009)      |              |              |
| np.log(pl)     | 0.044        |              |              | 1.748**      | 1.457**      |
|                | (0.106)      |              |              | (0.756)      | (0.715)      |
| np.log(pk)     | 0.188**      |              | 0.178**      | 1.247**      |              |
|                | (0.087)      |              | (0.083)      | (0.627)      |              |
| np.log(pf)     | 0.713***     |              | 0.720***     | 0.194        |              |
|                | (0.064)      |              | (0.062)      | (0.466)      |              |
| Intercept      | -7.472***    | -3.841***    | -7.061***    | -18.572**    | -10.002      |
|                | (1.063)      | (0.107)      | (0.402)      | (7.670)      | (6.413)      |
| R-squared      | 0.982        | 0.965        | 0.982        | 0.053        | 0.026        |
| R-squared Adj. | 0.982        | 0.965        | 0.982        | 0.035        | 0.020        |
| Observations   | 158          | 158          | 158          | 158          | 158          |

*Signif. codes:* `***` $p<0.01$, `**` $p<0.05$, `*` $p<0.1$. В скобках — неробастные стандартные ошибки.



In [6]:
#| output: asis
show_conditions(sign_level, ERRORS,
                'каждая регрессия незначима, т.е. все её коэффициенты, кроме константы, равны нулю')



::: {.callout-important title="Условия теста" icon="true"}
- уровень значимости: **1%**
- стандартные ошибки: **неробастные и робастные (HC3) — два варианта F-теста**
- гипотеза: каждая регрессия незначима, т.е. все её коэффициенты, кроме константы, равны нулю
:::



Вычислите тестовые статистики неробастного и робастного F-теста и критические значения. Какие из регрессий значимы? **Ответ округлите до 2-х десятичных знаков.**

In [7]:
#| output: asis
show_regression_tests(formulas, Electricity, sign_level)



| Регрессия   |       F |   F (HC3) |   F крит. | Вывод     | Вывод (HC3)   |
|:------------|--------:|----------:|----------:|:----------|:--------------|
| (1)         | 2106.93 |    552.67 |      3.44 | значима   | значима       |
| (2)         | 4359.64 |   1750.83 |      6.80 | значима   | значима       |
| (3)         | 2824.32 |    666.82 |      3.91 | значима   | значима       |
| (4)         |    2.88 |      2.30 |      3.91 | незначима | незначима     |
| (5)         |    4.16 |      3.32 |      6.80 | незначима | незначима     |



## sleep equation #2

Для набора данных `sleep75` рассмотрим несколько регрессий.

In [8]:
#| output: asis
sign_level = 0.01
formulas = [
    'sleep ~ totwrk + age + I(age**2) + smsa + yngkid + marr + union',
    'sleep ~ age + I(age**2) + smsa + yngkid + marr + union',
    'sleep ~ age + I(age**2)',
    'sleep ~ smsa + yngkid + marr + union']
mod = [fit_ols(f, sleep75) for f in formulas]
names = [f'({i})' for i in range(1, len(mod) + 1)]
show_models(mod, names, regressor_order=const_last(mod), info=RSE)



|                     | (1)         | (2)         | (3)         | (4)         |
|:--------------------|:------------|:------------|:------------|:------------|
| Dep. Variable       | sleep       | sleep       | sleep       | sleep       |
| totwrk              | -0.147***   |             |             |             |
|                     | (0.017)     |             |             |             |
| age                 | -8.800      | -22.785*    | -21.490*    |             |
|                     | (11.428)    | (11.907)    | (11.737)    |             |
| I(age ** 2)         | 0.142       | 0.317**     | 0.301**     |             |
|                     | (0.136)     | (0.141)     | (0.140)     |             |
| smsa                | -69.437**   | -57.707*    |             | -57.890*    |
|                     | (32.422)    | (34.088)    |             | (34.293)    |
| yngkid              | 1.631       | -3.184      |             | -31.138     |
|                     | (50.691)    | (53.338)    |             | (50.424)    |
| marr                | 47.551      | 62.147      |             | 60.098      |
|                     | (42.008)    | (44.169)    |             | (44.167)    |
| union               | 13.492      | 16.228      |             | 7.327       |
|                     | (38.369)    | (40.374)    |             | (40.530)    |
| Intercept           | 3673.756*** | 3601.335*** | 3608.030*** | 3242.521*** |
|                     | (224.507)   | (236.085)   | (230.646)   | (44.044)    |
| R-squared           | 0.118       | 0.022       | 0.015       | 0.007       |
| R-squared Adj.      | 0.110       | 0.014       | 0.012       | 0.002       |
| Observations        | 706         | 706         | 706         | 706         |
| Residual Std. Error | 419.371     | 441.301     | 441.775     | 444.018     |

*Signif. codes:* `***` $p<0.01$, `**` $p<0.05$, `*` $p<0.1$. В скобках — неробастные стандартные ошибки.



In [9]:
#| output: asis
show_conditions(sign_level, ERRORS,
                'каждая регрессия незначима, т.е. все её коэффициенты, кроме константы, равны нулю')



::: {.callout-important title="Условия теста" icon="true"}
- уровень значимости: **1%**
- стандартные ошибки: **неробастные и робастные (HC3) — два варианта F-теста**
- гипотеза: каждая регрессия незначима, т.е. все её коэффициенты, кроме константы, равны нулю
:::



Вычислите тестовые статистики неробастного и робастного F-теста и критические значения. Какие из регрессий значимы? **Ответ округлите до 2-х десятичных знаков.**

In [10]:
#| output: asis
show_regression_tests(formulas, sleep75, sign_level)



| Регрессия   |     F |   F (HC3) |   F крит. | Вывод     | Вывод (HC3)   |
|:------------|------:|----------:|----------:|:----------|:--------------|
| (1)         | 13.39 |     11.16 |      2.66 | значима   | значима       |
| (2)         |  2.66 |      3.27 |      2.83 | незначима | значима       |
| (3)         |  5.22 |      6.63 |      4.64 | значима   | значима       |
| (4)         |  1.31 |      1.40 |      3.35 | незначима | незначима     |



## cost equation #2

Для набора данных `Electricity` рассмотрим несколько регрессий.

In [11]:
#| output: asis
sign_level = 0.01
formulas = [
    'np.log(cost) ~ np.log(q) + I(np.log(q)**2) + np.log(pl) + np.log(pk) + np.log(pf)',
    'np.log(cost) ~ np.log(q) + I(np.log(q)**2)',
    'np.log(cost) ~ np.log(q) + np.log(pl) + np.log(pk) + np.log(pf)',
    'np.log(cost) ~ np.log(pl) + np.log(pk) + np.log(pf)',
    'np.log(cost) ~ np.log(pl)']
mod = [fit_ols(f, Electricity) for f in formulas]
names = [f'({i})' for i in range(1, len(mod) + 1)]
show_models(mod, names, regressor_order=const_last(mod), info=RSE)



|                     | (1)          | (2)          | (3)          | (4)          | (5)          |
|:--------------------|:-------------|:-------------|:-------------|:-------------|:-------------|
| Dep. Variable       | np.log(cost) | np.log(cost) | np.log(cost) | np.log(cost) | np.log(cost) |
| np.log(q)           | 0.403***     | 0.390***     | 0.838***     |              |              |
|                     | (0.032)      | (0.054)      | (0.009)      |              |              |
| I(np.log(q) ** 2)   | 0.030***     | 0.031***     |              |              |              |
|                     | (0.002)      | (0.004)      |              |              |              |
| np.log(pl)          | 0.146**      |              | 0.044        | 1.748**      | 1.457**      |
|                     | (0.070)      |              | (0.106)      | (0.756)      | (0.715)      |
| np.log(pk)          | 0.157***     |              | 0.188**      | 1.247**      |              |
|                     | (0.058)      |              | (0.087)      | (0.627)      |              |
| np.log(pf)          | 0.685***     |              | 0.713***     | 0.194        |              |
|                     | (0.043)      |              | (0.064)      | (0.466)      |              |
| Intercept           | -6.739***    | -2.394***    | -7.472***    | -18.572**    | -10.002      |
|                     | (0.706)      | (0.194)      | (1.063)      | (7.670)      | (6.413)      |
| R-squared           | 0.992        | 0.976        | 0.982        | 0.053        | 0.026        |
| R-squared Adj.      | 0.992        | 0.976        | 0.982        | 0.035        | 0.020        |
| Observations        | 158          | 158          | 158          | 158          | 158          |
| Residual Std. Error | 0.138        | 0.239        | 0.209        | 1.516        | 1.527        |

*Signif. codes:* `***` $p<0.01$, `**` $p<0.05$, `*` $p<0.1$. В скобках — неробастные стандартные ошибки.



In [12]:
#| output: asis
show_conditions(sign_level, ERRORS,
                'каждая регрессия незначима, т.е. все её коэффициенты, кроме константы, равны нулю')



::: {.callout-important title="Условия теста" icon="true"}
- уровень значимости: **1%**
- стандартные ошибки: **неробастные и робастные (HC3) — два варианта F-теста**
- гипотеза: каждая регрессия незначима, т.е. все её коэффициенты, кроме константы, равны нулю
:::



Вычислите тестовые статистики неробастного и робастного F-теста и критические значения. Какие из регрессий значимы? **Ответ округлите до 2-х десятичных знаков.**

In [13]:
#| output: asis
show_regression_tests(formulas, Electricity, sign_level)



| Регрессия   |       F |   F (HC3) |   F крит. | Вывод     | Вывод (HC3)   |
|:------------|--------:|----------:|----------:|:----------|:--------------|
| (1)         | 3880.41 |   2691.55 |      3.14 | значима   | значима       |
| (2)         | 3185.77 |   2815.03 |      4.74 | значима   | значима       |
| (3)         | 2106.93 |    552.67 |      3.44 | значима   | значима       |
| (4)         |    2.88 |      2.30 |      3.91 | незначима | незначима     |
| (5)         |    4.16 |      3.32 |      6.80 | незначима | незначима     |



# F-тест: совместная значимость

## sleep equation #3

Для набора данных `sleep75` рассмотрим регрессию

`sleep ~ totwrk + age + male + south + smsa + yngkid + marr + union`

Результаты оценивания:

In [14]:
#| output: asis
sign_level = 0.05
mod, mod_hc = fit_pair('sleep ~ totwrk + age + male + south + smsa + yngkid + marr + union', sleep75)
print_regression_summary(mod)
display(Markdown(nobs_text(mod)))



**Результаты $t$-теста для коэффициентов (неробастные стандартные ошибки):**

|           |    Coef. |   Std.Err. |   t-value |   p-value | Signif.   |
|:----------|---------:|-----------:|----------:|----------:|:----------|
| Intercept | 3446.830 |     81.840 |    42.117 |    <0.001 |           |
| totwrk    |   -0.169 |      0.018 |    -9.358 |    <0.001 | ***       |
| age       |    2.714 |      1.472 |     1.844 |     0.066 | *         |
| male      |   87.108 |     35.173 |     2.477 |     0.014 | **        |
| south     |  102.272 |     41.925 |     2.439 |     0.015 | **        |
| smsa      |  -54.188 |     33.193 |    -1.632 |     0.103 |           |
| yngkid    |  -13.051 |     50.459 |    -0.259 |     0.796 |           |
| marr      |   31.360 |     42.263 |     0.742 |     0.458 |           |
| union     |   11.866 |     38.186 |     0.311 |     0.756 |           |

*Signif. codes:* `***` $p<0.01$, `**` $p<0.05$, `*` $p<0.1$



Модель была подогнана по 706 наблюдениям.

In [15]:
#| output: asis
show_conditions(sign_level, ERRORS_T,
                r'$H_0:\beta_{smsa}=\beta_{yngkid}=\beta_{marr}=\beta_{union}=0$ '
                '(совместная значимость `smsa`, `yngkid`, `marr`, `union`)')



::: {.callout-important title="Условия теста" icon="true"}
- уровень значимости: **5%**
- стандартные ошибки: **неробастные и робастные (HC3) — два варианта F-теста; t-тест — неробастные**
- гипотеза: $H_0:\beta_{smsa}=\beta_{yngkid}=\beta_{marr}=\beta_{union}=0$ (совместная значимость `smsa`, `yngkid`, `marr`, `union`)
:::



Какие коэффициенты значимы по неробастным стандартным ошибкам?

In [16]:
#| output: asis
print_significant_coeffs(mod, sign_level)



::: {.callout-note title="Коэффициенты, значимые на уровне 5% (по неробастным стандартным ошибкам)" icon="true"}
`totwrk`, `male`, `south`
:::



Проверим совместную значимость `smsa`, `yngkid`, `marr`, `union`.

Вычислите критическое значение. **Ответ округлите до 3-х десятичных знаков.**

In [17]:
#| output: asis
null_hypothesis = 'smsa = 0, yngkid = 0, marr = 0, union = 0'
F_cr = show_f_crit(mod, null_hypothesis, sign_level)



::: {.callout-note title="Критическое значение" icon="true"}
$F_{crit} = 2.385$
:::



**Неробастный F-тест**

Вычислите тестовую статистику и P-значение. **Ответ округлите до 3-х десятичных знаков.**

In [18]:
#| output: asis
F_stat = show_f_test(mod, null_hypothesis)



|     F |   df1 |   df2 |   p-value |
|------:|------:|------:|----------:|
| 0.910 |     4 |   697 |     0.458 |



**Робастный F-тест (HC3)**

Вычислите тестовую статистику и P-значение. **Ответ округлите до 3-х десятичных знаков.**

In [19]:
#| output: asis
F_stat_hc = show_f_test(mod_hc, null_hypothesis)



|     F |   df1 |   df2 |   p-value |
|------:|------:|------:|----------:|
| 0.951 |     4 |   697 |     0.434 |



Значимо ли совместное влияние?

In [20]:
#| output: asis
show_conclusion((F_stat, F_stat_hc), F_cr, 'совместное влияние значимо', 'совместное влияние незначимо')



::: {.callout-tip title="Вывод" icon="true"}
- Неробастный F-тест: $F = 0.910 < 2.385$, гипотеза $H_0$ не отвергается: совместное влияние незначимо.
- Робастный F-тест (HC3): $F = 0.951 < 2.385$, гипотеза $H_0$ не отвергается: совместное влияние незначимо.
:::



## sleep equation #4

Для набора данных `sleep75` рассмотрим регрессию

`sleep ~ totwrk + age + I(age**2) + male + south`

Результаты оценивания:

In [21]:
#| output: asis
sign_level = 0.01
mod, mod_hc = fit_pair('sleep ~ totwrk + age + I(age**2) + male + south', sleep75)
print_regression_summary(mod)
display(Markdown(nobs_text(mod)))



**Результаты $t$-теста для коэффициентов (неробастные стандартные ошибки):**

|             |    Coef. |   Std.Err. |   t-value |   p-value | Signif.   |
|:------------|---------:|-----------:|----------:|----------:|:----------|
| Intercept   | 3613.995 |    218.433 |    16.545 |    <0.001 |           |
| totwrk      |   -0.167 |      0.018 |    -9.228 |    <0.001 | ***       |
| age         |   -6.255 |     11.191 |    -0.559 |     0.576 |           |
| I(age ** 2) |    0.109 |      0.134 |     0.814 |     0.416 |           |
| male        |   90.457 |     34.257 |     2.641 |     0.008 | ***       |
| south       |  114.547 |     40.637 |     2.819 |     0.005 | ***       |

*Signif. codes:* `***` $p<0.01$, `**` $p<0.05$, `*` $p<0.1$



Модель была подогнана по 706 наблюдениям.

In [22]:
#| output: asis
show_conditions(sign_level, ERRORS_T,
                r'$H_0:\beta_{age}=\beta_{age^2}=0$ (значимость влияния возраста)')



::: {.callout-important title="Условия теста" icon="true"}
- уровень значимости: **1%**
- стандартные ошибки: **неробастные и робастные (HC3) — два варианта F-теста; t-тест — неробастные**
- гипотеза: $H_0:\beta_{age}=\beta_{age^2}=0$ (значимость влияния возраста)
:::



Какие коэффициенты значимы по неробастным стандартным ошибкам?

In [23]:
#| output: asis
print_significant_coeffs(mod, sign_level)



::: {.callout-note title="Коэффициенты, значимые на уровне 1% (по неробастным стандартным ошибкам)" icon="true"}
`totwrk`, `male`, `south`
:::



Проверим значимость влияния возраста.

Вычислите критическое значение. **Ответ округлите до 3-х десятичных знаков.**

In [24]:
#| output: asis
null_hypothesis = 'age = 0, I(age ** 2) = 0'
F_cr = show_f_crit(mod, null_hypothesis, sign_level)



::: {.callout-note title="Критическое значение" icon="true"}
$F_{crit} = 4.636$
:::



**Неробастный F-тест**

Вычислите тестовую статистику и P-значение. **Ответ округлите до 3-х десятичных знаков.**

In [25]:
#| output: asis
F_stat = show_f_test(mod, null_hypothesis)



|     F |   df1 |   df2 |   p-value |
|------:|------:|------:|----------:|
| 2.347 |     2 |   700 |     0.096 |



**Робастный F-тест (HC3)**

Вычислите тестовую статистику и P-значение. **Ответ округлите до 3-х десятичных знаков.**

In [26]:
#| output: asis
F_stat_hc = show_f_test(mod_hc, null_hypothesis)



|     F |   df1 |   df2 |   p-value |
|------:|------:|------:|----------:|
| 2.908 |     2 |   700 |     0.055 |



Значимо ли совместное влияние?

In [27]:
#| output: asis
show_conclusion((F_stat, F_stat_hc), F_cr, 'совместное влияние значимо', 'совместное влияние незначимо')



::: {.callout-tip title="Вывод" icon="true"}
- Неробастный F-тест: $F = 2.347 < 4.636$, гипотеза $H_0$ не отвергается: совместное влияние незначимо.
- Робастный F-тест (HC3): $F = 2.908 < 4.636$, гипотеза $H_0$ не отвергается: совместное влияние незначимо.
:::



## sleep equation #5

Для набора данных `sleep75` рассмотрим регрессию

`sleep ~ totwrk + age + I(age**2) + male + south + totwrk:male`

Результаты оценивания:

In [28]:
#| output: asis
sign_level = 0.01
mod, mod_hc = fit_pair('sleep ~ totwrk + age + I(age**2) + male + south + totwrk:male', sleep75)
print_regression_summary(mod)
display(Markdown(nobs_text(mod)))



**Результаты $t$-теста для коэффициентов (неробастные стандартные ошибки):**

|             |    Coef. |   Std.Err. |   t-value |   p-value | Signif.   |
|:------------|---------:|-----------:|----------:|----------:|:----------|
| Intercept   | 3587.178 |    220.238 |    16.288 |    <0.001 |           |
| totwrk      |   -0.149 |      0.026 |    -5.673 |    <0.001 | ***       |
| age         |   -6.259 |     11.192 |    -0.559 |     0.576 |           |
| I(age ** 2) |    0.107 |      0.134 |     0.796 |     0.426 |           |
| male        |  161.890 |     82.177 |     1.970 |     0.049 | **        |
| south       |  112.144 |     40.717 |     2.754 |     0.006 | ***       |
| totwrk:male |   -0.035 |      0.036 |    -0.956 |     0.339 |           |

*Signif. codes:* `***` $p<0.01$, `**` $p<0.05$, `*` $p<0.1$



Модель была подогнана по 706 наблюдениям.

In [29]:
#| output: asis
show_conditions(sign_level, ERRORS_T,
                r'$H_0:\beta_{male}=\beta_{totwrk:male}=0$ (значимость влияния гендерной переменной)')



::: {.callout-important title="Условия теста" icon="true"}
- уровень значимости: **1%**
- стандартные ошибки: **неробастные и робастные (HC3) — два варианта F-теста; t-тест — неробастные**
- гипотеза: $H_0:\beta_{male}=\beta_{totwrk:male}=0$ (значимость влияния гендерной переменной)
:::



Какие коэффициенты значимы по неробастным стандартным ошибкам?

In [30]:
#| output: asis
print_significant_coeffs(mod, sign_level)



::: {.callout-note title="Коэффициенты, значимые на уровне 1% (по неробастным стандартным ошибкам)" icon="true"}
`totwrk`, `south`
:::



Проверим значимость влияния гендерной переменной.

Вычислите критическое значение. **Ответ округлите до 3-х десятичных знаков.**

In [31]:
#| output: asis
null_hypothesis = 'male = 0, totwrk:male = 0'
F_cr = show_f_crit(mod, null_hypothesis, sign_level)



::: {.callout-note title="Критическое значение" icon="true"}
$F_{crit} = 4.636$
:::



**Неробастный F-тест**

Вычислите тестовую статистику и P-значение. **Ответ округлите до 3-х десятичных знаков.**

In [32]:
#| output: asis
F_stat = show_f_test(mod, null_hypothesis)



|     F |   df1 |   df2 |   p-value |
|------:|------:|------:|----------:|
| 3.943 |     2 |   699 |     0.020 |



**Робастный F-тест (HC3)**

Вычислите тестовую статистику и P-значение. **Ответ округлите до 3-х десятичных знаков.**

In [33]:
#| output: asis
F_stat_hc = show_f_test(mod_hc, null_hypothesis)



|     F |   df1 |   df2 |   p-value |
|------:|------:|------:|----------:|
| 3.690 |     2 |   699 |     0.025 |



Значимо ли совместное влияние?

In [34]:
#| output: asis
show_conclusion((F_stat, F_stat_hc), F_cr, 'совместное влияние значимо', 'совместное влияние незначимо')



::: {.callout-tip title="Вывод" icon="true"}
- Неробастный F-тест: $F = 3.943 < 4.636$, гипотеза $H_0$ не отвергается: совместное влияние незначимо.
- Робастный F-тест (HC3): $F = 3.690 < 4.636$, гипотеза $H_0$ не отвергается: совместное влияние незначимо.
:::



## wage equation #1

Для набора данных `wage2` рассмотрим регрессию

`np.log(wage) ~ age + I(age**2) + IQ + south + urban + married`

Переменные: `wage` — месячная заработная плата; `age` — возраст, лет; `IQ` — результат теста IQ; `south`, `urban`, `married` — бинарные переменные.

Результаты оценивания:

In [35]:
#| output: asis
sign_level = 0.01
mod, mod_hc = fit_pair('np.log(wage) ~ age + I(age**2) + IQ + south + urban + married', wage2)
print_regression_summary(mod)
display(Markdown(nobs_text(mod)))



**Результаты $t$-теста для коэффициентов (неробастные стандартные ошибки):**

|             |   Coef. |   Std.Err. |   t-value |   p-value | Signif.   |
|:------------|--------:|-----------:|----------:|----------:|:----------|
| Intercept   |   2.938 |      1.656 |     1.774 |     0.076 |           |
| age         |   0.145 |      0.100 |     1.447 |     0.148 |           |
| I(age ** 2) |  -0.002 |      0.002 |    -1.236 |     0.217 |           |
| IQ          |   0.008 |      0.001 |     9.770 |    <0.001 | ***       |
| south       |  -0.100 |      0.027 |    -3.736 |    <0.001 | ***       |
| urban       |   0.173 |      0.028 |     6.260 |    <0.001 | ***       |
| married     |   0.202 |      0.040 |     5.008 |    <0.001 | ***       |

*Signif. codes:* `***` $p<0.01$, `**` $p<0.05$, `*` $p<0.1$



Модель была подогнана по 935 наблюдениям.

In [36]:
#| output: asis
show_conditions(sign_level, ERRORS_T,
                r'$H_0:\beta_{age}=\beta_{age^2}=0$ (значимость влияния возраста)')



::: {.callout-important title="Условия теста" icon="true"}
- уровень значимости: **1%**
- стандартные ошибки: **неробастные и робастные (HC3) — два варианта F-теста; t-тест — неробастные**
- гипотеза: $H_0:\beta_{age}=\beta_{age^2}=0$ (значимость влияния возраста)
:::



Какие коэффициенты значимы по неробастным стандартным ошибкам?

In [37]:
#| output: asis
print_significant_coeffs(mod, sign_level)



::: {.callout-note title="Коэффициенты, значимые на уровне 1% (по неробастным стандартным ошибкам)" icon="true"}
`IQ`, `south`, `urban`, `married`
:::



Проверим значимость влияния возраста.

Вычислите критическое значение. **Ответ округлите до 3-х десятичных знаков.**

In [38]:
#| output: asis
null_hypothesis = 'age = 0, I(age ** 2) = 0'
F_cr = show_f_crit(mod, null_hypothesis, sign_level)



::: {.callout-note title="Критическое значение" icon="true"}
$F_{crit} = 4.628$
:::



**Неробастный F-тест**

Вычислите тестовую статистику и P-значение. **Ответ округлите до 3-х десятичных знаков.**

In [39]:
#| output: asis
F_stat = show_f_test(mod, null_hypothesis)



|      F |   df1 |   df2 |   p-value |
|-------:|------:|------:|----------:|
| 14.833 |     2 |   928 |    <0.001 |



**Робастный F-тест (HC3)**

Вычислите тестовую статистику и P-значение. **Ответ округлите до 3-х десятичных знаков.**

In [40]:
#| output: asis
F_stat_hc = show_f_test(mod_hc, null_hypothesis)



|      F |   df1 |   df2 |   p-value |
|-------:|------:|------:|----------:|
| 14.700 |     2 |   928 |    <0.001 |



Значимо ли совместное влияние?

In [41]:
#| output: asis
show_conclusion((F_stat, F_stat_hc), F_cr, 'совместное влияние значимо', 'совместное влияние незначимо')



::: {.callout-tip title="Вывод" icon="true"}
- Неробастный F-тест: $F = 14.833 > 4.628$, гипотеза $H_0$ отвергается: совместное влияние значимо.
- Робастный F-тест (HC3): $F = 14.700 > 4.628$, гипотеза $H_0$ отвергается: совместное влияние значимо.
:::



# F-тест: структурные ограничения

## cost equation #3

Для набора данных `Electricity` рассмотрим регрессию

```
np.log(cost) ~ np.log(q) + I(np.log(q)**2) +
               np.log(pf) + np.log(pl) + np.log(pk)
```

Результаты оценивания:

In [42]:
#| output: asis
sign_level = 0.01
mod, mod_hc = fit_pair(
    'np.log(cost) ~ np.log(q) + I(np.log(q)**2) + np.log(pf) + np.log(pl) + np.log(pk)', Electricity)
show_models([mod], ['(1)'], regressor_order=const_last([mod]), info=FULL_STATS)



|                     | (1)          |
|:--------------------|:-------------|
| Dep. Variable       | np.log(cost) |
| np.log(q)           | 0.403***     |
|                     | (0.032)      |
| I(np.log(q) ** 2)   | 0.030***     |
|                     | (0.002)      |
| np.log(pf)          | 0.685***     |
|                     | (0.043)      |
| np.log(pl)          | 0.146**      |
|                     | (0.070)      |
| np.log(pk)          | 0.157***     |
|                     | (0.058)      |
| Intercept           | -6.739***    |
|                     | (0.706)      |
| R-squared           | 0.992        |
| R-squared Adj.      | 0.992        |
| Observations        | 158          |
| Residual Std. Error | 0.138        |
| F Statistic         | 3880.407***  |

*Signif. codes:* `***` $p<0.01$, `**` $p<0.05$, `*` $p<0.1$. В скобках — неробастные стандартные ошибки.



In [43]:
#| output: asis
show_conditions(sign_level, ERRORS_T,
                [r'$H_0:\beta_{pf}+\beta_{pl}+\beta_{pk}=1$',
                 r'$H_0:\beta_{pl}=\beta_{pk}$',
                 r'$H_0:\beta_{pf}=\beta_{pl}=\beta_{pk}$'],
                extra=[r'обозначения: $\beta_{pf}$, $\beta_{pl}$, $\beta_{pk}$ — коэффициенты '
                       'при `np.log(pf)`, `np.log(pl)`, `np.log(pk)`'])



::: {.callout-important title="Условия теста" icon="true"}
- уровень значимости: **1%**
- стандартные ошибки: **неробастные и робастные (HC3) — два варианта F-теста; t-тест — неробастные**
- гипотеза 1: $H_0:\beta_{pf}+\beta_{pl}+\beta_{pk}=1$
- гипотеза 2: $H_0:\beta_{pl}=\beta_{pk}$
- гипотеза 3: $H_0:\beta_{pf}=\beta_{pl}=\beta_{pk}$
- обозначения: $\beta_{pf}$, $\beta_{pl}$, $\beta_{pk}$ — коэффициенты при `np.log(pf)`, `np.log(pl)`, `np.log(pk)`
:::



Какие коэффициенты значимы по неробастным стандартным ошибкам?

In [44]:
#| output: asis
print_significant_coeffs(mod, sign_level)



::: {.callout-note title="Коэффициенты, значимые на уровне 1% (по неробастным стандартным ошибкам)" icon="true"}
`np.log(q)`, `I(np.log(q) ** 2)`, `np.log(pf)`, `np.log(pk)`
:::



Проверим гипотезы 1–3. Какая интерпретация у каждой гипотезы?

In [45]:
#| output: asis
display_answer('\n'.join([
    '- Гипотеза 1: функция издержек однородна первой степени по ценам факторов — '
    'если все цены вырастут на 1%, издержки вырастут на 1%.',
    '- Гипотеза 2: эластичности издержек по цене труда и по цене капитала одинаковы.',
    '- Гипотеза 3: эластичности издержек по ценам всех трёх факторов одинаковы.']),
    theme='tip', title='Ответ')



::: {.callout-tip title="Ответ" icon="true"}
- Гипотеза 1: функция издержек однородна первой степени по ценам факторов — если все цены вырастут на 1%, издержки вырастут на 1%.
- Гипотеза 2: эластичности издержек по цене труда и по цене капитала одинаковы.
- Гипотеза 3: эластичности издержек по ценам всех трёх факторов одинаковы.
:::



Для каждой гипотезы вычислите тестовую статистику и P-значение неробастного и робастного F-теста. **Ответ округлите до 3-х десятичных знаков.**

**Гипотеза 1. Неробастный F-тест**

In [46]:
#| output: asis
tests = []
null_hypothesis = 'np.log(pf) + np.log(pl) + np.log(pk) = 1'
F_stat = show_f_test(mod, null_hypothesis)



|     F |   df1 |   df2 |   p-value |
|------:|------:|------:|----------:|
| 0.015 |     1 |   152 |     0.904 |



**Гипотеза 1. Робастный F-тест (HC3)**

In [47]:
#| output: asis
tests.append((null_hypothesis, F_stat, show_f_test(mod_hc, null_hypothesis)))



|     F |   df1 |   df2 |   p-value |
|------:|------:|------:|----------:|
| 0.013 |     1 |   152 |     0.909 |



**Гипотеза 2. Неробастный F-тест**

In [48]:
#| output: asis
null_hypothesis = 'np.log(pl) - np.log(pk) = 0'
F_stat = show_f_test(mod, null_hypothesis)



|     F |   df1 |   df2 |   p-value |
|------:|------:|------:|----------:|
| 0.020 |     1 |   152 |     0.887 |



**Гипотеза 2. Робастный F-тест (HC3)**

In [49]:
#| output: asis
tests.append((null_hypothesis, F_stat, show_f_test(mod_hc, null_hypothesis)))



|     F |   df1 |   df2 |   p-value |
|------:|------:|------:|----------:|
| 0.017 |     1 |   152 |     0.896 |



**Гипотеза 3. Неробастный F-тест**

In [50]:
#| output: asis
null_hypothesis = 'np.log(pf) - np.log(pk) = 0, np.log(pl) - np.log(pk) = 0'
F_stat = show_f_test(mod, null_hypothesis)



|      F |   df1 |   df2 |   p-value |
|-------:|------:|------:|----------:|
| 26.417 |     2 |   152 |    <0.001 |



**Гипотеза 3. Робастный F-тест (HC3)**

In [51]:
#| output: asis
tests.append((null_hypothesis, F_stat, show_f_test(mod_hc, null_hypothesis)))



|      F |   df1 |   df2 |   p-value |
|-------:|------:|------:|----------:|
| 15.834 |     2 |   152 |    <0.001 |



Вычислите критические значения. Какой можно сделать вывод по каждой гипотезе? **Ответ округлите до 3-х десятичных знаков.**

In [52]:
#| output: asis
show_conclusions(mod, tests, sign_level)



::: {.callout-tip title="Критические значения и выводы" icon="true"}
- Гипотеза 1: $F_{crit} = 6.805$.
    - Неробастный F-тест: $F = 0.015 < 6.805$, гипотеза $H_0$ не отвергается.
    - Робастный F-тест (HC3): $F = 0.013 < 6.805$, гипотеза $H_0$ не отвергается.
- Гипотеза 2: $F_{crit} = 6.805$.
    - Неробастный F-тест: $F = 0.020 < 6.805$, гипотеза $H_0$ не отвергается.
    - Робастный F-тест (HC3): $F = 0.017 < 6.805$, гипотеза $H_0$ не отвергается.
- Гипотеза 3: $F_{crit} = 4.748$.
    - Неробастный F-тест: $F = 26.417 > 4.748$, гипотеза $H_0$ отвергается.
    - Робастный F-тест (HC3): $F = 15.834 > 4.748$, гипотеза $H_0$ отвергается.
:::



## output equation

Для набора данных `Labour` рассмотрим регрессию **np.log(output) на np.log(capital), np.log(labour), np.log(wage)**.

Переменные: `output` — добавленная стоимость, млн евро; `capital` — основные средства, млн евро; `labour` — число работников; `wage` — затраты на оплату труда на одного работника, тыс. евро.

Результаты оценивания:

In [53]:
#| output: asis
sign_level = 0.05
mod, mod_hc = fit_pair('np.log(output) ~ np.log(capital) + np.log(labour) + np.log(wage)', Labour)
show_models([mod], ['(1)'], regressor_order=const_last([mod]), info=FULL_STATS)



|                     | (1)            |
|:--------------------|:---------------|
| Dep. Variable       | np.log(output) |
| np.log(capital)     | 0.149***       |
|                     | (0.015)        |
| np.log(labour)      | 0.720***       |
|                     | (0.019)        |
| np.log(wage)        | 0.921***       |
|                     | (0.058)        |
| Intercept           | -5.007***      |
|                     | (0.221)        |
| R-squared           | 0.888          |
| R-squared Adj.      | 0.888          |
| Observations        | 569            |
| Residual Std. Error | 0.397          |
| F Statistic         | 1499.235***    |

*Signif. codes:* `***` $p<0.01$, `**` $p<0.05$, `*` $p<0.1$. В скобках — неробастные стандартные ошибки.



In [54]:
#| output: asis
show_conditions(sign_level, ERRORS_T,
                [r'$H_0:\beta_{capital}+\beta_{labour}+\beta_{wage}=1$',
                 r'$H_0:\beta_{labour}=\beta_{wage}$',
                 r'$H_0:\beta_{capital}=\beta_{labour}=\beta_{wage}$'],
                extra=[r'обозначения: $\beta_{capital}$, $\beta_{labour}$, $\beta_{wage}$ — коэффициенты '
                       'при `np.log(capital)`, `np.log(labour)`, `np.log(wage)`'])



::: {.callout-important title="Условия теста" icon="true"}
- уровень значимости: **5%**
- стандартные ошибки: **неробастные и робастные (HC3) — два варианта F-теста; t-тест — неробастные**
- гипотеза 1: $H_0:\beta_{capital}+\beta_{labour}+\beta_{wage}=1$
- гипотеза 2: $H_0:\beta_{labour}=\beta_{wage}$
- гипотеза 3: $H_0:\beta_{capital}=\beta_{labour}=\beta_{wage}$
- обозначения: $\beta_{capital}$, $\beta_{labour}$, $\beta_{wage}$ — коэффициенты при `np.log(capital)`, `np.log(labour)`, `np.log(wage)`
:::



Какие коэффициенты значимы по неробастным стандартным ошибкам?

In [55]:
#| output: asis
print_significant_coeffs(mod, sign_level)



::: {.callout-note title="Коэффициенты, значимые на уровне 5% (по неробастным стандартным ошибкам)" icon="true"}
`np.log(capital)`, `np.log(labour)`, `np.log(wage)`
:::



Проверим гипотезы 1–3. Какая интерпретация у каждой гипотезы?

In [56]:
#| output: asis
display_answer('\n'.join([
    '- Гипотеза 1: сумма эластичностей выпуска по капиталу, труду и зарплате равна единице.',
    '- Гипотеза 2: эластичности выпуска по числу занятых и по зарплате одинаковы.',
    '- Гипотеза 3: все три эластичности выпуска — по капиталу, труду и зарплате — одинаковы.']),
    theme='tip', title='Ответ')



::: {.callout-tip title="Ответ" icon="true"}
- Гипотеза 1: сумма эластичностей выпуска по капиталу, труду и зарплате равна единице.
- Гипотеза 2: эластичности выпуска по числу занятых и по зарплате одинаковы.
- Гипотеза 3: все три эластичности выпуска — по капиталу, труду и зарплате — одинаковы.
:::



Для каждой гипотезы вычислите тестовую статистику и P-значение неробастного и робастного F-теста. **Ответ округлите до 3-х десятичных знаков.**

**Гипотеза 1. Неробастный F-тест**

In [57]:
#| output: asis
tests = []
null_hypothesis = 'np.log(capital) + np.log(labour) + np.log(wage) = 1'
F_stat = show_f_test(mod, null_hypothesis)



|       F |   df1 |   df2 |   p-value |
|--------:|------:|------:|----------:|
| 198.587 |     1 |   565 |    <0.001 |



**Гипотеза 1. Робастный F-тест (HC3)**

In [58]:
#| output: asis
tests.append((null_hypothesis, F_stat, show_f_test(mod_hc, null_hypothesis)))



|       F |   df1 |   df2 |   p-value |
|--------:|------:|------:|----------:|
| 109.340 |     1 |   565 |    <0.001 |



**Гипотеза 2. Неробастный F-тест**

In [59]:
#| output: asis
null_hypothesis = 'np.log(wage) - np.log(labour) = 0'
F_stat = show_f_test(mod, null_hypothesis)



|      F |   df1 |   df2 |   p-value |
|-------:|------:|------:|----------:|
| 11.081 |     1 |   565 |    <0.001 |



**Гипотеза 2. Робастный F-тест (HC3)**

In [60]:
#| output: asis
tests.append((null_hypothesis, F_stat, show_f_test(mod_hc, null_hypothesis)))



|     F |   df1 |   df2 |   p-value |
|------:|------:|------:|----------:|
| 4.289 |     1 |   565 |     0.039 |



**Гипотеза 3. Неробастный F-тест**

In [61]:
#| output: asis
null_hypothesis = 'np.log(capital) - np.log(labour) = 0, np.log(labour) - np.log(wage) = 0'
F_stat = show_f_test(mod, null_hypothesis)



|       F |   df1 |   df2 |   p-value |
|--------:|------:|------:|----------:|
| 192.367 |     2 |   565 |    <0.001 |



**Гипотеза 3. Робастный F-тест (HC3)**

In [62]:
#| output: asis
tests.append((null_hypothesis, F_stat, show_f_test(mod_hc, null_hypothesis)))



|      F |   df1 |   df2 |   p-value |
|-------:|------:|------:|----------:|
| 47.863 |     2 |   565 |    <0.001 |



Вычислите критические значения. Какой можно сделать вывод по каждой гипотезе? **Ответ округлите до 3-х десятичных знаков.**

In [63]:
#| output: asis
show_conclusions(mod, tests, sign_level)



::: {.callout-tip title="Критические значения и выводы" icon="true"}
- Гипотеза 1: $F_{crit} = 3.858$.
    - Неробастный F-тест: $F = 198.587 > 3.858$, гипотеза $H_0$ отвергается.
    - Робастный F-тест (HC3): $F = 109.340 > 3.858$, гипотеза $H_0$ отвергается.
- Гипотеза 2: $F_{crit} = 3.858$.
    - Неробастный F-тест: $F = 11.081 > 3.858$, гипотеза $H_0$ отвергается.
    - Робастный F-тест (HC3): $F = 4.289 > 3.858$, гипотеза $H_0$ отвергается.
- Гипотеза 3: $F_{crit} = 3.012$.
    - Неробастный F-тест: $F = 192.367 > 3.012$, гипотеза $H_0$ отвергается.
    - Робастный F-тест (HC3): $F = 47.863 > 3.012$, гипотеза $H_0$ отвергается.
:::



# F-тест: структурные сдвиги

## sleep equation #6

Для набора данных `sleep75` рассмотрим регрессию `sleep ~ totwrk + age + I(age**2) + south + smsa + marr` со структурными сдвигами относительно гендерной бинарной переменной `male`.

Результаты оценивания модели со структурными сдвигами:

In [64]:
#| output: asis
sign_level = 0.01
mod, mod_hc = fit_pair(
    'sleep ~ totwrk + age + I(age**2) + south + smsa + marr + male + male:totwrk + male:age'
    ' + male:I(age**2) + male:south + male:smsa + male:marr', sleep75)
show_coefs(mod)



| Переменная       |   Оценка |
|:-----------------|---------:|
| Intercept        | 3933.485 |
| totwrk           |   -0.140 |
| age              |  -26.641 |
| I(age ** 2)      |    0.345 |
| south            |  129.190 |
| smsa             |  -33.660 |
| marr             |   63.314 |
| male             | -415.456 |
| male:totwrk      |   -0.049 |
| male:age         |   33.969 |
| male:I(age ** 2) |   -0.393 |
| male:south       |  -67.588 |
| male:smsa        |  -43.849 |
| male:marr        |  -55.125 |

Модель была подогнана по 706 наблюдениям.



In [65]:
#| output: asis
show_conditions(sign_level, ERRORS,
                '$H_0$: коэффициенты при `male` и всех произведениях с `male` равны нулю '
                '(структурных сдвигов нет)')



::: {.callout-important title="Условия теста" icon="true"}
- уровень значимости: **1%**
- стандартные ошибки: **неробастные и робастные (HC3) — два варианта F-теста**
- гипотеза: $H_0$: коэффициенты при `male` и всех произведениях с `male` равны нулю (структурных сдвигов нет)
:::



Проверьте значимость структурных сдвигов. **Ответ округлите до 3-х десятичных знаков.**

Вычислите критическое значение.

In [66]:
#| output: asis
null_hypothesis = shift_hyp(mod, 'male')
F_cr = show_f_crit(mod, null_hypothesis, sign_level)



::: {.callout-note title="Критическое значение" icon="true"}
$F_{crit} = 2.665$
:::



**Неробастный F-тест**

Вычислите тестовую статистику и P-значение.

In [67]:
#| output: asis
F_stat = show_f_test(mod, null_hypothesis)



|     F |   df1 |   df2 |   p-value |
|------:|------:|------:|----------:|
| 1.551 |     7 |   692 |     0.147 |



**Робастный F-тест (HC3)**

Вычислите тестовую статистику и P-значение.

In [68]:
#| output: asis
F_stat_hc = show_f_test(mod_hc, null_hypothesis)



|     F |   df1 |   df2 |   p-value |
|------:|------:|------:|----------:|
| 1.530 |     7 |   692 |     0.154 |



Значимы ли структурные сдвиги?

In [69]:
#| output: asis
show_conclusion((F_stat, F_stat_hc), F_cr, 'структурные сдвиги значимы', 'структурные сдвиги незначимы')



::: {.callout-tip title="Вывод" icon="true"}
- Неробастный F-тест: $F = 1.551 < 2.665$, гипотеза $H_0$ не отвергается: структурные сдвиги незначимы.
- Робастный F-тест (HC3): $F = 1.530 < 2.665$, гипотеза $H_0$ не отвергается: структурные сдвиги незначимы.
:::



## sleep equation #7

Для набора данных `sleep75` рассмотрим регрессию `sleep ~ totwrk + age + I(age**2) + male + smsa` со структурными сдвигами относительно географической бинарной переменной `south`.

Результаты оценивания модели со структурными сдвигами:

In [70]:
#| output: asis
sign_level = 0.01
mod, mod_hc = fit_pair(
    'sleep ~ totwrk + age + I(age**2) + male + smsa + south + south:totwrk + south:age'
    ' + south:I(age**2) + male:south + south:smsa', sleep75)
show_coefs(mod)



| Переменная        |   Оценка |
|:------------------|---------:|
| Intercept         | 3663.564 |
| totwrk            |   -0.183 |
| age               |   -4.720 |
| I(age ** 2)       |    0.071 |
| male              |  117.058 |
| smsa              |  -66.366 |
| south             |  -55.855 |
| south:totwrk      |    0.107 |
| south:age         |  -10.403 |
| south:I(age ** 2) |    0.229 |
| male:south        | -116.776 |
| south:smsa        |   57.077 |

Модель была подогнана по 706 наблюдениям.



In [71]:
#| output: asis
show_conditions(sign_level, ERRORS,
                '$H_0$: коэффициенты при `south` и всех произведениях с `south` равны нулю '
                '(структурных сдвигов нет)')



::: {.callout-important title="Условия теста" icon="true"}
- уровень значимости: **1%**
- стандартные ошибки: **неробастные и робастные (HC3) — два варианта F-теста**
- гипотеза: $H_0$: коэффициенты при `south` и всех произведениях с `south` равны нулю (структурных сдвигов нет)
:::



Проверьте значимость структурных сдвигов. **Ответ округлите до 3-х десятичных знаков.**

Вычислите критическое значение.

In [72]:
#| output: asis
null_hypothesis = shift_hyp(mod, 'south')
F_cr = show_f_crit(mod, null_hypothesis, sign_level)



::: {.callout-note title="Критическое значение" icon="true"}
$F_{crit} = 2.828$
:::



**Неробастный F-тест**

Вычислите тестовую статистику и P-значение.

In [73]:
#| output: asis
F_stat = show_f_test(mod, null_hypothesis)



|     F |   df1 |   df2 |   p-value |
|------:|------:|------:|----------:|
| 2.649 |     6 |   694 |     0.015 |



**Робастный F-тест (HC3)**

Вычислите тестовую статистику и P-значение.

In [74]:
#| output: asis
F_stat_hc = show_f_test(mod_hc, null_hypothesis)



|     F |   df1 |   df2 |   p-value |
|------:|------:|------:|----------:|
| 2.101 |     6 |   694 |     0.051 |



Значимы ли структурные сдвиги?

In [75]:
#| output: asis
show_conclusion((F_stat, F_stat_hc), F_cr, 'структурные сдвиги значимы', 'структурные сдвиги незначимы')



::: {.callout-tip title="Вывод" icon="true"}
- Неробастный F-тест: $F = 2.649 < 2.828$, гипотеза $H_0$ не отвергается: структурные сдвиги незначимы.
- Робастный F-тест (HC3): $F = 2.101 < 2.828$, гипотеза $H_0$ не отвергается: структурные сдвиги незначимы.
:::



## wage equation #2

Для набора данных `wage1` рассмотрим регрессию `np.log(wage) ~ exper + I(exper**2) + married + smsa` со структурными сдвигами относительно гендерной бинарной переменной `female`. Переменные: `wage` — средняя почасовая заработная плата; `exper` — потенциальный опыт работы, лет; `married`, `smsa`, `female` — бинарные переменные.

Результаты оценивания модели со структурными сдвигами:

In [76]:
#| output: asis
sign_level = 0.01
mod, mod_hc = fit_pair(
    'np.log(wage) ~ exper + I(exper**2) + married + smsa + female + female:exper'
    ' + female:I(exper**2) + female:married + female:smsa', wage1)
show_coefs(mod)



| Переменная           |   Оценка |
|:---------------------|---------:|
| Intercept            |    1.079 |
| exper                |    0.044 |
| I(exper ** 2)        |   -0.001 |
| married              |    0.289 |
| smsa                 |    0.292 |
| female               |   -0.007 |
| female:exper         |   -0.018 |
| female:I(exper ** 2) | 0.000359 |
| female:married       |   -0.280 |
| female:smsa          |   -0.071 |

Модель была подогнана по 526 наблюдениям.



In [77]:
#| output: asis
show_conditions(sign_level, ERRORS,
                '$H_0$: коэффициенты при `female` и всех произведениях с `female` равны нулю '
                '(структурных сдвигов нет)')



::: {.callout-important title="Условия теста" icon="true"}
- уровень значимости: **1%**
- стандартные ошибки: **неробастные и робастные (HC3) — два варианта F-теста**
- гипотеза: $H_0$: коэффициенты при `female` и всех произведениях с `female` равны нулю (структурных сдвигов нет)
:::



Проверьте значимость структурных сдвигов. **Ответ округлите до 3-х десятичных знаков.**

Вычислите критическое значение.

In [78]:
#| output: asis
null_hypothesis = shift_hyp(mod, 'female')
F_cr = show_f_crit(mod, null_hypothesis, sign_level)



::: {.callout-note title="Критическое значение" icon="true"}
$F_{crit} = 3.053$
:::



**Неробастный F-тест**

Вычислите тестовую статистику и P-значение.

In [79]:
#| output: asis
F_stat = show_f_test(mod, null_hypothesis)



|      F |   df1 |   df2 |   p-value |
|-------:|------:|------:|----------:|
| 21.254 |     5 |   516 |    <0.001 |



**Робастный F-тест (HC3)**

Вычислите тестовую статистику и P-значение.

In [80]:
#| output: asis
F_stat_hc = show_f_test(mod_hc, null_hypothesis)



|      F |   df1 |   df2 |   p-value |
|-------:|------:|------:|----------:|
| 23.266 |     5 |   516 |    <0.001 |



Значимы ли структурные сдвиги?

In [81]:
#| output: asis
show_conclusion((F_stat, F_stat_hc), F_cr, 'структурные сдвиги значимы', 'структурные сдвиги незначимы')



::: {.callout-tip title="Вывод" icon="true"}
- Неробастный F-тест: $F = 21.254 > 3.053$, гипотеза $H_0$ отвергается: структурные сдвиги значимы.
- Робастный F-тест (HC3): $F = 23.266 > 3.053$, гипотеза $H_0$ отвергается: структурные сдвиги значимы.
:::



## wage equation #3

Для набора данных `wage1` рассмотрим регрессию `np.log(wage) ~ exper + I(exper**2) + female + smsa`. Она оценена по всем наблюдениям, только по состоящим в браке (`married = 1`) и только по не состоящим в браке (`married = 0`). Результаты приведены в таблице.

In [82]:
#| output: asis
spec = 'np.log(wage) ~ exper + I(exper**2) + female + smsa'
parts = [fit_ols(spec, wage1), fit_ols(spec, wage1[wage1['married'] == 1]),
         fit_ols(spec, wage1[wage1['married'] == 0])]
info = {**N_OBS, 'RSS': lambda m: f'{m.ssr:.3f}',
        **{key: FULL_STATS[key] for key in ('Residual Std. Error', 'F Statistic')}}
show_models_exact(parts, ['все', 'в браке', 'не в браке'],
                  regressor_order=parts[0].params.index, info=info)



|                     | все          | в браке      | не в браке   |
|:--------------------|:-------------|:-------------|:-------------|
| Dep. Variable       | np.log(wage) | np.log(wage) | np.log(wage) |
| Intercept           | 1.293***     | 1.588***     | 1.076***     |
|                     | (0.060)      | (0.088)      | (0.089)      |
| exper               | 0.044***     | 0.025***     | 0.045***     |
|                     | (0.005)      | (0.008)      | (0.008)      |
| I(exper ** 2)       | -0.001***    | -0.001***    | -0.001***    |
|                     | (0.000116)   | (0.000158)   | (0.000195)   |
| female              | -0.382***    | -0.496***    | -0.168**     |
|                     | (0.040)      | (0.049)      | (0.065)      |
| smsa                | 0.264***     | 0.259***     | 0.276***     |
|                     | (0.044)      | (0.052)      | (0.077)      |
| R-squared           | 0.280        | 0.318        | 0.188        |
| R-squared Adj.      | 0.274        | 0.309        | 0.172        |
| F Statistic         | 50.589***    | 36.702***    | 11.637***    |
| Observations        | 526          | 320          | 206          |
| RSS                 | 106.835      | 58.616       | 41.839       |
| Residual Std. Error | 0.453        | 0.431        | 0.456        |

*Signif. codes:* `***` $p<0.01$, `**` $p<0.05$, `*` $p<0.1$. В скобках — неробастные стандартные ошибки.



По значениям RSS вычислите F-статистику теста Чоу для структурного сдвига по переменной `married`. **Ответ округлите до 3-х десятичных знаков.**

In [83]:
#| output: asis
rss, rss_1, rss_2 = (m.ssr for m in parts)
k, n = len(parts[0].params), int(parts[0].nobs)
F_chow = ((rss - rss_1 - rss_2) / k) / ((rss_1 + rss_2) / (n - 2 * k))
display_answer(
    rf'$F = \dfrac{{({rss:.3f} - {rss_1:.3f} - {rss_2:.3f})/{k}}}'
    rf'{{({rss_1:.3f} + {rss_2:.3f})/{n - 2 * k}}} = {fmt(F_chow)}$',
    theme='tip', title='Ответ')



::: {.callout-tip title="Ответ" icon="true"}
$F = \dfrac{(106.835 - 58.616 - 41.839)/5}{(58.616 + 41.839)/516} = 6.554$
:::



Оцените модель со структурными сдвигами относительно бинарной переменной `married`.

Результаты оценивания модели со структурными сдвигами:

In [84]:
#| output: asis
sign_level = 0.01
mod, mod_hc = fit_pair(
    'np.log(wage) ~ exper + I(exper**2) + female + smsa + married + married:exper'
    ' + married:I(exper**2) + female:married + married:smsa', wage1)
show_coefs(mod)



| Переменная            |   Оценка |
|:----------------------|---------:|
| Intercept             |    1.076 |
| exper                 |    0.045 |
| I(exper ** 2)         |   -0.001 |
| female                |   -0.168 |
| smsa                  |    0.276 |
| married               |    0.511 |
| married:exper         |   -0.020 |
| married:I(exper ** 2) | 0.000334 |
| female:married        |   -0.328 |
| married:smsa          |   -0.017 |

Модель была подогнана по 526 наблюдениям.



In [85]:
#| output: asis
show_conditions(sign_level, ERRORS,
                '$H_0$: коэффициенты при `married` и всех произведениях с `married` равны нулю '
                '(структурных сдвигов нет)')



::: {.callout-important title="Условия теста" icon="true"}
- уровень значимости: **1%**
- стандартные ошибки: **неробастные и робастные (HC3) — два варианта F-теста**
- гипотеза: $H_0$: коэффициенты при `married` и всех произведениях с `married` равны нулю (структурных сдвигов нет)
:::



Проверьте значимость структурных сдвигов. **Ответ округлите до 3-х десятичных знаков.**

Вычислите критическое значение.

In [86]:
#| output: asis
null_hypothesis = shift_hyp(mod, 'married')
F_cr = show_f_crit(mod, null_hypothesis, sign_level)



::: {.callout-note title="Критическое значение" icon="true"}
$F_{crit} = 3.053$
:::



**Неробастный F-тест**

Вычислите тестовую статистику и P-значение.

In [87]:
#| output: asis
F_stat = show_f_test(mod, null_hypothesis)



|     F |   df1 |   df2 |   p-value |
|------:|------:|------:|----------:|
| 6.554 |     5 |   516 |    <0.001 |



**Робастный F-тест (HC3)**

Вычислите тестовую статистику и P-значение.

In [88]:
#| output: asis
F_stat_hc = show_f_test(mod_hc, null_hypothesis)



|     F |   df1 |   df2 |   p-value |
|------:|------:|------:|----------:|
| 7.984 |     5 |   516 |    <0.001 |



Значимы ли структурные сдвиги? Совпадает ли неробастная статистика со статистикой теста Чоу?

In [89]:
#| output: asis
assert abs(F_stat - F_chow) < 1e-8
show_conclusion((F_stat, F_stat_hc), F_cr, 'структурные сдвиги значимы', 'структурные сдвиги незначимы',
                extra=[f'Неробастная F-статистика совпадает со статистикой теста Чоу: {fmt(F_chow)}.'])



::: {.callout-tip title="Вывод" icon="true"}
- Неробастный F-тест: $F = 6.554 > 3.053$, гипотеза $H_0$ отвергается: структурные сдвиги значимы.
- Робастный F-тест (HC3): $F = 7.984 > 3.053$, гипотеза $H_0$ отвергается: структурные сдвиги значимы.
- Неробастная F-статистика совпадает со статистикой теста Чоу: 6.554.
:::

